In [1]:
from pathlib import Path
from contextlib import closing
import json
import re
import sqlite3

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW, PROCESSED, DB = ROOT / "data/raw", ROOT / "data/processed", ROOT / "data/coffee.db"
TABLES = ROOT / "docs/raport-cw2/tables"
TABLES.mkdir(parents=True, exist_ok=True)
START, END = 2000, 2023
pd.set_option("display.max_rows", 80, "display.max_colwidth", 200)


def query(sql):
    with closing(sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True)) as con:
        return pd.read_sql(sql, con)


def number(value, decimals=0):
    return f"{value:,.{decimals}f}".replace(",", " ").replace(".", ",")


def save_table(frame, name):
    (TABLES / f"{name}.md").write_text(frame.to_markdown(index=False) + "\n", encoding="utf-8")
    display(frame)

In [2]:
manifest = json.loads((RAW / "sources.json").read_text(encoding="utf-8"))
qcl = pd.read_csv(PROCESSED / "faostat_coffee_all.csv", dtype={"Area Code (M49)": "string"})
tcl = pd.read_csv(PROCESSED / "faostat_trade_coffee.csv", dtype={"Area Code (M49)": "string"})
era5 = json.loads((RAW / manifest["worldbank"]["path"]).read_text(encoding="utf-8"))["data"]
era5_years = [int(timestamp[:4]) for series in era5["tas"].values() for timestamp in series]
pink_sheet = pd.read_excel(RAW / manifest["pink_sheet"]["path"], "Annual Prices (Nominal)", header=6, skiprows=[7])
cpi = json.loads((RAW / manifest["cpi_usa"]["path"]).read_text(encoding="utf-8"))[1]
m49_table = re.search(r'<table\b[^>]*id\s*=\s*"downloadTableEN"[^>]*>(.*?)</table>',
                      (RAW / manifest["un_m49"]["path"]).read_text(encoding="utf-8"), re.S).group(1)
m49_iso3 = re.findall(r"<td[^>]*>\s*([A-Z]{3})\s*</td>", m49_table)
m49_columns = max(len(re.findall(r"<td\b", row)) for row in re.findall(r"<tr\b[^>]*>(.*?)</tr>", m49_table, re.S))
ico = pd.read_excel(RAW / manifest["ico_production"]["path"], header=None)
ico_countries = int(ico[1].astype(str).str.fullmatch(r"\((A|R|A/R|R/A)\)").sum())
optimum = pd.read_csv(RAW / manifest["weather_optimum"]["path"])


def years(values):
    return f"{min(values)}–{max(values)}"


def fao_areas(frame):
    return frame.loc[frame["Area Code"].lt(5000), "Area"].nunique()


sources = pd.DataFrame([
    ["FAOSTAT QCL – zbiory (wycinek: kawa)", "CSV w ZIP", f"{number(len(qcl))} × {qcl.shape[1]}",
     "Area Code + Element + Year", years(qcl["Year"]), fao_areas(qcl), "t, ha, kg/ha"],
    ["FAOSTAT TCL – eksport (wycinek: kawa)", "CSV w ZIP", f"{number(len(tcl))} × {tcl.shape[1]}",
     "Area Code + Element Code + Year", years(tcl["Year"]), fao_areas(tcl), "t, 1000 USD"],
    ["World Bank CCKP / ERA5 – pogoda", "JSON (API)", f"{len(era5['tas'])} × {len(set(era5_years))} × {len(era5)}",
     "ISO3 + rok + zmienna", years(era5_years), len(era5["tas"]), "°C, mm"],
    ["World Bank Pink Sheet – ceny", "XLSX", f"{len(pink_sheet)} × {pink_sheet.shape[1]}",
     "rok", years(pink_sheet.iloc[:, 0]), "świat", "USD/kg"],
    ["World Bank WDI – CPI USA", "JSON (API)", f"{len(cpi)} × 2",
     "rok", years([int(row["date"]) for row in cpi]), "USA", "indeks, 2010 = 100"],
    ["UN M49 – kody krajów", "HTML", f"{len(m49_iso3)} × {m49_columns}", "kod M49 → ISO3", "–", len(m49_iso3), "–"],
    ["ICO – gatunek kawy", "XLSX", f"{ico_countries} × {ico.shape[1]}", "nazwa kraju",
     f"{ico.iloc[3, 2]}–{ico.iloc[3, ico.shape[1] - 1]}", ico_countries, "typ A, R, A/R, R/A"],
    ["Literatura – optimum pogody", "CSV (ręcznie)", f"{len(optimum)} × {optimum.shape[1]}", "gatunek", "–", "–", "°C, mm"],
], columns=["źródło", "format", "rozmiar (wiersze × kolumny)", "klucz", "lata", "kraje", "jednostki"])
save_table(sources, "sources")

C:\Users\chame\studia\sem7\systemy-analizy-biznesowej\projekt\systemy-analizy-biznesowej-projekt\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,źródło,format,rozmiar (wiersze × kolumny),klucz,lata,kraje,jednostki
0,FAOSTAT QCL – zbiory (wycinek: kawa),CSV w ZIP,20 209 × 14,Area Code + Element + Year,1961–2024,86,"t, ha, kg/ha"
1,FAOSTAT TCL – eksport (wycinek: kawa),CSV w ZIP,25 219 × 9,Area Code + Element Code + Year,1961–2024,204,"t, 1000 USD"
2,World Bank CCKP / ERA5 – pogoda,JSON (API),246 × 76 × 2,ISO3 + rok + zmienna,1950–2025,246,"°C, mm"
3,World Bank Pink Sheet – ceny,XLSX,66 × 70,rok,1960–2025,świat,USD/kg
4,World Bank WDI – CPI USA,JSON (API),24 × 2,rok,2000–2023,USA,"indeks, 2010 = 100"
5,UN M49 – kody krajów,HTML,248 × 16,kod M49 → ISO3,–,248,–
6,ICO – gatunek kawy,XLSX,54 × 32,nazwa kraju,1990/91–2019/20,54,"typ A, R, A/R, R/A"
7,Literatura – optimum pogody,CSV (ręcznie),2 × 6,gatunek,–,–,"°C, mm"


In [3]:
model = query("""
    SELECT k.iso3, k.year, k.production_t, k.yield_kg_ha, k.k1_export_t AS export_t,
           k.k2_export_value_usd AS export_value_usd, k.k3_usd_per_kg AS usd_per_kg,
           k.k6_usd_2023_per_kg AS usd_2023_per_kg, k.temperature_c, k.precipitation_mm,
           y.arabica_usd_kg, y.robusta_usd_kg
    FROM v_kpi_country_year AS k JOIN dim_year AS y USING (year)""")


def short(value):
    return number(value, 2) if isinstance(value, float) else value


profile = pd.DataFrame([{
    "kolumna": column,
    "typ": "liczba" if pd.api.types.is_numeric_dtype(values) else "tekst",
    "braki [%]": round(values.isna().mean() * 100, 1),
    "unikalne": values.nunique(),
    "min": short(values.min()),
    "mediana": short(float(values.median())) if pd.api.types.is_numeric_dtype(values) else "–",
    "max": short(values.max()),
} for column, values in model.items()])
save_table(profile, "profile")

,kolumna,typ,braki [%],unikalne,min,mediana,max
0,iso3,tekst,0.0,51,BDI,–,ZWE
1,year,liczba,0.0,24,2000,"2 011,50",2023
2,production_t,liczba,0.0,1165,"1,18","32 035,00","3 705 719,00"
3,yield_kg_ha,liczba,0.0,1132,"82,20","647,20","3 915,80"
4,export_t,liczba,0.0,1165,"0,00","11 141,27","2 372 632,78"
5,export_value_usd,liczba,0.0,1163,"0,00","18 212 000,00","8 514 139 000,00"
6,usd_per_kg,liczba,1.7,1186,"0,29","2,67","125,00"
7,usd_2023_per_kg,liczba,1.7,1203,"0,36","3,58","155,38"
8,temperature_c,liczba,0.0,644,"6,63","24,20","27,75"
9,precipitation_mm,liczba,0.0,1222,"93,16","1 570,54","4 897,40"


In [4]:
RULES = [
    ("R1", "(iso3, year) unikalne i niepuste", "krytyczny", "schemat: PRIMARY KEY",
     "SELECT iso3, year FROM fact_country_year GROUP BY iso3, year HAVING COUNT(*) > 1 OR iso3 IS NULL OR year IS NULL"),
    ("R2", "iso3 i year z faktów istnieją w wymiarach", "krytyczny", "schemat: FOREIGN KEY",
     """SELECT f.iso3, f.year FROM fact_country_year AS f
        LEFT JOIN dim_country AS c USING (iso3) LEFT JOIN dim_year AS y USING (year)
        WHERE c.iso3 IS NULL OR y.year IS NULL"""),
    ("R3", "zbiory i eksport w t, wartość w 1000 USD", "krytyczny", "assert na kolumnie Unit przed ładowaniem", None),
    ("R4", "zbiory, eksport i wartość ≥ 0", "krytyczny", "schemat: CHECK",
     "SELECT iso3, year FROM fact_country_year WHERE production_t < 0 OR export_t < 0 OR export_value_usd < 0"),
    ("R5", "eksport (wolumen i wartość) we wszystkich 24 latach", "wysoki", "notebook 02: wykluczenie kraju",
     "SELECT iso3, NULL AS year FROM fact_country_year GROUP BY iso3 HAVING COUNT(export_t) < 24 OR COUNT(export_value_usd) < 24"),
    ("R6", "każdy rok ma CPI i obie ceny", "wysoki", "schemat: NOT NULL",
     "SELECT NULL AS iso3, year FROM dim_year WHERE cpi_usa IS NULL OR arabica_usd_kg IS NULL OR robusta_usd_kg IS NULL"),
    ("R7", "K3 tylko przy wolumenie > 0; K4 i K5 tylko przy podstawie > 0", "wysoki", "widok: NULLIF",
     """SELECT iso3, year FROM (
            SELECT *, LAG(k1_export_t) OVER w AS previous_t, LAG(k2_export_value_usd) OVER w AS previous_usd
            FROM v_kpi_country_year WINDOW w AS (PARTITION BY iso3 ORDER BY year))
        WHERE (k1_export_t = 0 AND k3_usd_per_kg IS NOT NULL)
           OR (previous_t = 0 AND k4_export_t_yoy_pct IS NOT NULL)
           OR (previous_usd = 0 AND k5_export_value_yoy_pct IS NOT NULL)"""),
    ("R8", "zmiana r/r zbiorów lub eksportu powyżej ±50%", "średni", "zapytanie kontrolne → flaga",
     "SELECT iso3, year FROM v_kpi_country_year WHERE ABS(production_yoy_pct) > 50 OR ABS(k4_export_t_yoy_pct) > 50"),
    ("R9", "eksport większy niż zbiory", "średni", "zapytanie kontrolne → flaga",
     "SELECT iso3, year FROM fact_country_year WHERE export_t > production_t"),
    ("R10", "K3 poza 0,5–2 × cena referencyjna gatunku (próg wstępny)", "średni", "zapytanie kontrolne → przegląd",
     "SELECT iso3, year FROM v_kpi_country_year WHERE k3_usd_per_kg NOT BETWEEN 0.5 * reference_usd_kg AND 2 * reference_usd_kg"),
    ("R11", "plon ≈ zbiory / powierzchnia (±5%)", "niski", "zapytanie kontrolne → flaga",
     "SELECT iso3, year FROM fact_country_year WHERE area_ha > 0 AND ABS(yield_kg_ha - production_t / area_ha * 1000) > 0.05 * yield_kg_ha"),
]
summary, flagged = [], []
for rule, description, priority, enforcement, sql in RULES:
    rows = query(sql) if sql else None
    if rows is not None and len(rows):
        flagged.append(rows.assign(rule=rule))
    summary.append({"ID": rule, "reguła": description, "priorytet": priority, "egzekwowanie": enforcement,
                    "naruszenia [wiersze]": "–" if rows is None else len(rows)})
rules = pd.DataFrame(summary)
blocking = rules.loc[rules["priorytet"].isin(["krytyczny", "wysoki"]) & rules["naruszenia [wiersze]"].ne("–")]
assert blocking["naruszenia [wiersze]"].eq(0).all(), blocking
countries = query("SELECT iso3, country FROM dim_country")
review = pd.concat(flagged).merge(countries, on="iso3", how="left")[["rule", "iso3", "country", "year"]]
review.to_csv(PROCESSED / "dq_review.csv", index=False)
save_table(rules, "dq_rules")
display(review.groupby("rule").agg(rows=("iso3", "size"), countries=("iso3", "nunique")))

,ID,reguła,priorytet,egzekwowanie,naruszenia [wiersze]
0,R1,"(iso3, year) unikalne i niepuste",krytyczny,schemat: PRIMARY KEY,0
1,R2,iso3 i year z faktów istnieją w wymiarach,krytyczny,schemat: FOREIGN KEY,0
2,R3,"zbiory i eksport w t, wartość w 1000 USD",krytyczny,assert na kolumnie Unit przed ładowaniem,–
3,R4,"zbiory, eksport i wartość ≥ 0",krytyczny,schemat: CHECK,0
4,R5,eksport (wolumen i wartość) we wszystkich 24 latach,wysoki,notebook 02: wykluczenie kraju,0
5,R6,każdy rok ma CPI i obie ceny,wysoki,schemat: NOT NULL,0
6,R7,K3 tylko przy wolumenie > 0; K4 i K5 tylko przy podstawie > 0,wysoki,widok: NULLIF,0
7,R8,zmiana r/r zbiorów lub eksportu powyżej ±50%,średni,zapytanie kontrolne → flaga,309
8,R9,eksport większy niż zbiory,średni,zapytanie kontrolne → flaga,152
9,R10,"K3 poza 0,5–2 × cena referencyjna gatunku (próg wstępny)",średni,zapytanie kontrolne → przegląd,148


,rows,countries
rule,,
R10,148,20
R11,1,1
R8,309,40
R9,152,25


In [5]:
fact = query("SELECT * FROM fact_country_year")
kpi = query("SELECT * FROM v_kpi_country_year")
kpi_country = query("SELECT * FROM v_kpi_country")
excluded = pd.read_csv(PROCESSED / "excluded_export_countries.csv")
n_countries = fact["iso3"].nunique()


def violations(rule):
    return rules.loc[rules["ID"].eq(rule), "naruszenia [wiersze]"].item()


def estimated(flags):
    return f"{flags.ne('A').mean() * 100:.0f}%"


zero_export = kpi.loc[kpi["k1_export_t"].eq(0)]
no_species = kpi_country.loc[kpi_country["species"].isna(), "country"]
never_in_optimum = (kpi_country.dropna(subset=["species"]).groupby("species")["k8_weather_in_optimum_pct"]
                    .agg(lambda s: f"{s.eq(0).sum()} z {len(s)}"))
usa = fact.loc[fact["iso3"].eq("USA"), ["export_t", "production_t"]].mean()
updated = {name: manifest[name]["last_updated"][:10] for name in ["faostat", "faostat_trade", "pink_sheet", "cpi_usa"]}

quality = pd.DataFrame([
    ["kompletność",
     f"{len(excluded)} z {len(excluded) + n_countries} producentów nie ma eksportu we wszystkich latach {START}–{END}; "
     f"największy udział wykluczonego kraju w światowym eksporcie to {number(excluded['max_world_export_share_pct'].max(), 2)}%",
     f"wykluczenie (R5), lista w excluded_export_countries.csv; w modelu zostaje {n_countries} krajów"],
    ["kompletność",
     f"lata z zerowym eksportem: {len(zero_export)}, kraje: {zero_export['iso3'].nunique()} "
     f"({', '.join(sorted(zero_export['country'].unique()))})",
     "K3–K6 bez wyniku w tych latach (R7)"],
    ["kompletność",
     f"kraje bez gatunku w ICO: {len(no_species)} z {n_countries} ({'; '.join(no_species)})",
     "K8 i K9 nie są liczone; Chiny – decyzja 8"],
    ["spójność",
     "nazwy krajów różnią się między FAO, M49 i ICO (np. „United Republic of Tanzania” i „Tanzania”)",
     f"łączenie po kodzie M49 – każdy z {n_countries} krajów ma kod M49 i ISO3; 4 aliasy dla nazw ICO"],
    ["spójność",
     "agregat FAO „China” (kod 351) obejmuje m.in. „China, mainland” i dublowałby dane",
     "agregat pominięty, używamy „China, mainland”"],
    ["spójność",
     "wartość eksportu w 1000 USD, ceny światowe w USD/kg, wszystko w cenach bieżących",
     "wartość ×1000; ceny stałe z 2023 r. liczone jednym indeksem CPI USA (K6)"],
    ["aktualność",
     f"model kończy się na {END} r.; aktualizacje źródeł: QCL {updated['faostat']}, TCL {updated['faostat_trade']}, "
     f"Pink Sheet {updated['pink_sheet']}, CPI {updated['cpi_usa']}; FAOSTAT ma już dane za {tcl['Year'].max()} r.",
     f"zostajemy przy zakresie {START}–{END} z etapu 1; opóźnienie ok. 2 lat nie przeszkadza w wyborze dostawców"],
    ["poprawność",
     f"udział wartości szacowanych lub imputowanych (flaga inna niż A): zbiory {estimated(fact['production_flag'])}, "
     f"plon {estimated(fact['yield_flag'])}, wolumen eksportu {estimated(fact['export_t_flag'])}, "
     f"wartość eksportu {estimated(fact['export_value_flag'])}",
     "flagi zostają w fact_country_year; w raporcie zaznaczamy niepewność"],
    ["poprawność",
     f"wiersze ze zmianą r/r powyżej ±50%: {violations('R8')} (zbiory: {kpi['production_yoy_pct'].abs().gt(50).sum()}, "
     f"eksport: {kpi['k4_export_t_yoy_pct'].abs().gt(50).sum()})",
     "flaga R8 bez usuwania; w porównaniach mediany i minimalny wolumen (decyzja 3)"],
    ["poprawność",
     f"wiersze z eksportem większym niż zbiory: {violations('R9')}, kraje: {review.loc[review['rule'].eq('R9'), 'iso3'].nunique()}; "
     f"USA eksportuje średnio {number(usa['export_t'] / usa['production_t'], 1)}× więcej, niż zbiera",
     "flaga R9; USA to reeksporter – wyłączyć z wyboru dostawców (decyzja 8)"],
    ["poprawność",
     f"wiersze z K3 poza przedziałem 0,5–2 × cena referencyjna gatunku: {violations('R10')}",
     "przegląd ręczny; ostateczny próg ustalimy w EDA"],
    ["poprawność",
     f"pogoda to średnia dla całego kraju; kraje, w których pogoda ani razu nie mieści się w optimum (K8 = 0): "
     f"arabika {never_in_optimum['arabica']}, robusta {never_in_optimum['robusta']}",
     "porównania w obrębie kraju; K8 i K9 względem normy 1991–2020 (decyzja 6)"],
    ["poprawność",
     f"wiersze, w których plon odbiega od zbiorów / powierzchni o ponad 5%: {violations('R11')}",
     "flaga R11"],
], columns=["wymiar", "obserwacja", "działanie naprawcze"])
save_table(quality, "quality")

,wymiar,obserwacja,działanie naprawcze
0,kompletność,"24 z 75 producentów nie ma eksportu we wszystkich latach 2000–2023; największy udział wykluczonego kraju w światowym eksporcie to 0,24%","wykluczenie (R5), lista w excluded_export_countries.csv; w modelu zostaje 51 krajów"
1,kompletność,"lata z zerowym eksportem: 21, kraje: 5 (Myanmar, New Caledonia, Philippines, Sri Lanka, Vanuatu)",K3–K6 bez wyniku w tych latach (R7)
2,kompletność,"kraje bez gatunku w ICO: 6 z 51 (China, mainland; Myanmar; Malaysia; New Caledonia; United States of America; Vanuatu)",K8 i K9 nie są liczone; Chiny – decyzja 8
3,spójność,"nazwy krajów różnią się między FAO, M49 i ICO (np. „United Republic of Tanzania” i „Tanzania”)",łączenie po kodzie M49 – każdy z 51 krajów ma kod M49 i ISO3; 4 aliasy dla nazw ICO
4,spójność,"agregat FAO „China” (kod 351) obejmuje m.in. „China, mainland” i dublowałby dane","agregat pominięty, używamy „China, mainland”"
5,spójność,"wartość eksportu w 1000 USD, ceny światowe w USD/kg, wszystko w cenach bieżących",wartość ×1000; ceny stałe z 2023 r. liczone jednym indeksem CPI USA (K6)
6,aktualność,"model kończy się na 2023 r.; aktualizacje źródeł: QCL 2025-12-31, TCL 2026-07-24, Pink Sheet 2026-09-02, CPI 2026-11-11; FAOSTAT ma już dane za 2024 r.",zostajemy przy zakresie 2000–2023 z etapu 1; opóźnienie ok. 2 lat nie przeszkadza w wyborze dostawców
7,poprawność,"udział wartości szacowanych lub imputowanych (flaga inna niż A): zbiory 22%, plon 44%, wolumen eksportu 25%, wartość eksportu 22%",flagi zostają w fact_country_year; w raporcie zaznaczamy niepewność
8,poprawność,"wiersze ze zmianą r/r powyżej ±50%: 309 (zbiory: 73, eksport: 260)",flaga R8 bez usuwania; w porównaniach mediany i minimalny wolumen (decyzja 3)
9,poprawność,"wiersze z eksportem większym niż zbiory: 152, kraje: 25; USA eksportuje średnio 12,8× więcej, niż zbiera",flaga R9; USA to reeksporter – wyłączyć z wyboru dostawców (decyzja 8)


In [6]:
FLAG = "flaga FAO: A oficjalna, E szacunek, I imputacja, X źródło zewnętrzne"
DESCRIPTIONS = {
    ("fact_country_year", "iso3"): ("–", "kod kraju ISO 3166-1 alfa-3", "UN M49"),
    ("fact_country_year", "year"): ("rok", "rok kalendarzowy", "–"),
    ("fact_country_year", "area_ha"): ("ha", "powierzchnia zbiorów kawy", "FAOSTAT QCL"),
    ("fact_country_year", "production_t"): ("t", "zbiory zielonej kawy", "FAOSTAT QCL"),
    ("fact_country_year", "yield_kg_ha"): ("kg/ha", "plon", "FAOSTAT QCL"),
    ("fact_country_year", "yield_deviation_pct"): ("%", "odchylenie plonu od liniowego trendu kraju 2000–2023", "obliczone (notebook 01)"),
    ("fact_country_year", "export_t"): ("t", "wolumen eksportu zielonej kawy (K1)", "FAOSTAT TCL"),
    ("fact_country_year", "export_value_usd"): ("USD", "wartość eksportu w cenach bieżących, przeliczona z 1000 USD (K2)", "FAOSTAT TCL"),
    ("fact_country_year", "temperature_c"): ("°C", "średnia roczna temperatura, średnia dla obszaru kraju", "CCKP / ERA5"),
    ("fact_country_year", "precipitation_mm"): ("mm", "roczna suma opadów, średnia dla obszaru kraju", "CCKP / ERA5"),
    ("fact_country_year", "temperature_anomaly_c"): ("°C", "różnica względem średniej kraju z lat 1991–2020", "obliczone (notebook 01)"),
    ("fact_country_year", "precipitation_anomaly_pct"): ("%", "różnica względem średniej kraju z lat 1991–2020", "obliczone (notebook 01)"),
    ("fact_country_year", "area_flag"): ("–", FLAG, "FAOSTAT QCL"),
    ("fact_country_year", "production_flag"): ("–", FLAG, "FAOSTAT QCL"),
    ("fact_country_year", "yield_flag"): ("–", FLAG, "FAOSTAT QCL"),
    ("fact_country_year", "export_t_flag"): ("–", FLAG, "FAOSTAT TCL"),
    ("fact_country_year", "export_value_flag"): ("–", FLAG, "FAOSTAT TCL"),
    ("dim_country", "iso3"): ("–", "kod kraju ISO 3166-1 alfa-3", "UN M49"),
    ("dim_country", "country"): ("–", "nazwa kraju według FAO", "FAOSTAT"),
    ("dim_country", "m49"): ("–", "kod kraju UN M49", "UN M49"),
    ("dim_country", "region"): ("–", "kontynent", "UN M49"),
    ("dim_country", "subregion"): ("–", "region pośredni M49 (np. South America), a gdy go brak – subregion", "UN M49"),
    ("dim_country", "ico_type"): ("–", "typ kawy: A arabika, R robusta, A/R głównie arabika, R/A głównie robusta", "ICO"),
    ("dim_country", "species"): ("–", "gatunek dominujący; brak dla krajów spoza listy ICO", "ICO"),
    ("dim_year", "year"): ("rok", "rok kalendarzowy", "–"),
    ("dim_year", "cpi_usa"): ("2010 = 100", "indeks cen konsumpcyjnych USA", "World Bank WDI"),
    ("dim_year", "to_usd_2023"): ("–", "mnożnik do cen stałych z 2023 r.: CPI(2023) / CPI(rok)", "obliczone"),
    ("dim_year", "arabica_usd_kg"): ("USD/kg", "średnia roczna cena arabiki, ceny bieżące", "World Bank Pink Sheet"),
    ("dim_year", "robusta_usd_kg"): ("USD/kg", "średnia roczna cena robusty, ceny bieżące", "World Bank Pink Sheet"),
    ("ref_weather_optimum", "species"): ("–", "gatunek kawy", "–"),
    ("ref_weather_optimum", "temperature_min_c"): ("°C", "dolna granica optymalnej średniej rocznej temperatury", "DaMatta i Ramalho (2006)"),
    ("ref_weather_optimum", "temperature_max_c"): ("°C", "górna granica optymalnej średniej rocznej temperatury", "DaMatta i Ramalho (2006)"),
    ("ref_weather_optimum", "precipitation_min_mm"): ("mm", "dolna granica optymalnej rocznej sumy opadów", "DaMatta i Ramalho (2006)"),
    ("ref_weather_optimum", "precipitation_max_mm"): ("mm", "górna granica optymalnej rocznej sumy opadów", "DaMatta i Ramalho (2006)"),
    ("ref_weather_optimum", "source"): ("–", "pełne źródło przedziałów", "–"),
    ("weather_yield_correlation", "iso3"): ("–", "kod kraju ISO 3166-1 alfa-3", "UN M49"),
    ("weather_yield_correlation", "yield_temperature_r"): ("r", "korelacja Pearsona plonu z temperaturą po usunięciu trendów (K7)", "obliczone (notebook 01)"),
    ("weather_yield_correlation", "yield_precipitation_r"): ("r", "korelacja Pearsona plonu z opadami po usunięciu trendów (K7)", "obliczone (notebook 01)"),
    ("weather_yield_correlation", "temperature_paired_years"): ("lata", "liczba lat użytych do korelacji z temperaturą", "obliczone (notebook 01)"),
    ("weather_yield_correlation", "precipitation_paired_years"): ("lata", "liczba lat użytych do korelacji z opadami", "obliczone (notebook 01)"),
}
dictionary = []
with closing(sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True)) as con:
    for table in ["fact_country_year", "dim_country", "dim_year", "ref_weather_optimum", "weather_yield_correlation"]:
        foreign_keys = {row[3]: f"FK → {row[2]}" for row in con.execute(f"PRAGMA foreign_key_list({table})")}
        for _, column, sql_type, not_null, _, primary_key in con.execute(f"PRAGMA table_info({table})"):
            unit, meaning, source = DESCRIPTIONS[table, column]
            constraints = ["PK" if primary_key else "", foreign_keys.get(column, ""),
                           "NOT NULL" if not_null and not primary_key else ""]
            dictionary.append({"tabela": table, "kolumna": column, "typ": sql_type, "jednostka": unit,
                               "znaczenie": meaning, "źródło": source, "ograniczenia": ", ".join(filter(None, constraints))})
dictionary = pd.DataFrame(dictionary)
assert set(zip(dictionary["tabela"], dictionary["kolumna"])) == set(DESCRIPTIONS)
save_table(dictionary, "data_dictionary")

,tabela,kolumna,typ,jednostka,znaczenie,źródło,ograniczenia
0,fact_country_year,iso3,TEXT,–,kod kraju ISO 3166-1 alfa-3,UN M49,"PK, FK → dim_country"
1,fact_country_year,year,INTEGER,rok,rok kalendarzowy,–,"PK, FK → dim_year"
2,fact_country_year,area_ha,REAL,ha,powierzchnia zbiorów kawy,FAOSTAT QCL,NOT NULL
3,fact_country_year,production_t,REAL,t,zbiory zielonej kawy,FAOSTAT QCL,NOT NULL
4,fact_country_year,yield_kg_ha,REAL,kg/ha,plon,FAOSTAT QCL,NOT NULL
5,fact_country_year,yield_deviation_pct,REAL,%,odchylenie plonu od liniowego trendu kraju 2000–2023,obliczone (notebook 01),NOT NULL
6,fact_country_year,export_t,REAL,t,wolumen eksportu zielonej kawy (K1),FAOSTAT TCL,NOT NULL
7,fact_country_year,export_value_usd,REAL,USD,"wartość eksportu w cenach bieżących, przeliczona z 1000 USD (K2)",FAOSTAT TCL,NOT NULL
8,fact_country_year,temperature_c,REAL,°C,"średnia roczna temperatura, średnia dla obszaru kraju",CCKP / ERA5,NOT NULL
9,fact_country_year,precipitation_mm,REAL,mm,"roczna suma opadów, średnia dla obszaru kraju",CCKP / ERA5,NOT NULL
